# P1-L4｜随机场景与期权风险

> **对应课程**：第 17–21 课  
> **模块**：P1-M4 · 随机模型与衍生品风险  
> **数据身份**：固定随机种子模拟教学数据，不是真实证券行情。

区分真实世界模拟、风险中性定价和局部对冲，检查模型假设变化的影响。

本实验向最终策略报告提供：**模型假设、情景风险与对冲敏感度**。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 分开真实世界情景与风险中性定价

左侧使用主观漂移模拟未来现价，右侧使用风险中性 Black–Scholes 价格；两种概率不能混称。

In [2]:
from src.quant_core import black_scholes_call, black_scholes_greeks

rng = np.random.default_rng(365)
spot, mu, sigma, maturity = 100.0, 0.06, 0.25, 0.5
terminal = spot * np.exp((mu - 0.5 * sigma**2) * maturity + sigma * np.sqrt(maturity) * rng.standard_normal(20_000))
strike = 105.0
physical_exercise_probability = np.mean(terminal > strike)
model_price = black_scholes_call(spot, strike, maturity, 0.02, sigma)
greeks = black_scholes_greeks(spot, strike, maturity, 0.02, sigma)

display(pd.Series({
    "physical_exercise_probability_under_mu": physical_exercise_probability,
    "risk_neutral_bs_price": model_price,
    **greeks,
}))

physical_exercise_probability_under_mu    0.423450
risk_neutral_bs_price                     5.376621
delta                                     0.447871
gamma                                     0.022375
theta_per_day                            -0.021316
vega_per_vol_point                        0.279683
dtype: float64

## 验收

- 关键计算能由小例子或独立公式核对；
- 数据时点、信号时点和成交时点没有混用；
- 结果同时包含金融含义和失败边界；
- 产物能够进入第 29 课策略报告和第 30 课能力矩阵。